# SERI — Spatial Effective Rainfall Index — Demo notebook

**Companion demo to the SERI v1.0 release.**

This notebook walks through the public API of the `seri` package and reproduces the headline numerical result of the concept paper:

> Selkh, C. (2026). *A Century After De Martonne: Why Spatial Coherence is the Missing Dimension of Aridity in the Hyper-Arid Sahara.* Earth-Science Reviews, in review.

It targets readers who want to compute SERI on their own events after a `pip install seri`.

## 0. Installation

```bash
pip install seri              # core API
pip install "seri[plot]"      # add matplotlib for figures
pip install "seri[earthengine]"  # add Google Earth Engine wrapper
```

In [ ]:
import seri
print(f"seri version: {seri.__version__}")
print(f"public API:   {sorted(seri.__all__)}")

## 1. The Abadla 2015 anchor case (manuscript § 5.1)

The concept paper anchors the framework on a single fully-documented event: the late-February 2015 regional rainfall episode in the lower Saoura basin near Abadla (Algerian Sahara, ~31 °N, ~−2.7 °E).

Reconstructed from GPM IMERG, the event delivered approximately:

| Quantity | Value | Unit |
|---|---|---|
| `P` (mean intensity) | 10.79 | mm |
| `A` (contiguous area) | 1 624 | km² |
| `f` (winter regime) | 1.30 | – |
| `g` (mixed reg + wadi-bottom) | 1.10 | – |
| `α` (working value) | 0.68 | – |

yielding **SERI ≈ 2 353** → *perennial-response* tier.

In [ ]:
result = seri.compute(
    P=10.79,           # mean intensity (mm)
    A=1624,            # contiguous area (km²)
    season=2,          # February → winter regime
    substrate="mixed", # area-weighted reg + wadi-bottom
)
result

In [ ]:
print(f"SERI value:   {result.value:.1f}   (published: ~2353)")
print(f"Tier:         {result.tier.name} — {result.tier_name}")
print(f"Description:  {result.tier_description}")

## 2. Diagnostic separation — Fig. 3 of the manuscript

Two events with **comparable peak intensity** can produce radically different ecological outcomes depending on their spatial coherence. This is the central claim of the paper, and the diagnostic value of SERI relative to scalar P-based indices.

In [ ]:
# Event A: summer convective cell — small footprint, high evaporative demand
event_a = seri.compute(P=18.0, A=51,   season=7, substrate="reg")

# Event B: winter frontal-system — large footprint (analogous to Abadla 2015)
event_b = seri.compute(P=17.0, A=1281, season=2, substrate="mixed")

print(f"Event A:  P=18 mm, A=51 km²    →  SERI = {event_a.value:>6.0f}  →  {event_a.tier_name}")
print(f"Event B:  P=17 mm, A=1281 km²  →  SERI = {event_b.value:>6.0f}  →  {event_b.tier_name}")

Two events with nearly identical scalar `P` are **separated by several tiers** by SERI, driven entirely by the explicit spatial-coherence term `A^α`. No scalar P-based index (SPI, SPEI, De Martonne, Emberger) makes this distinction.

## 3. Batch processing

For an event archive, use `compute_batch`:

In [ ]:
events = [
    {"P": 10.79, "A": 1624, "season":  2, "substrate": "mixed"},
    {"P":  8.0,  "A":  300, "season": 11, "substrate": "reg"},
    {"P": 18.0,  "A":   51, "season":  7, "substrate": "reg"},
    {"P": 25.0,  "A": 8000, "season":  3, "substrate": "wadi_bottom"},
    {"P":  3.0,  "A":  100, "season":  6, "substrate": "hamada"},
]

for ev, r in zip(events, seri.compute_batch(events)):
    print(f"P={ev['P']:>5.2f} mm  A={ev['A']:>5} km²  m={ev['season']:<2}  "
          f"sub={ev['substrate']:<13}  SERI={r.value:>7.1f}  →  {r.tier_name}")

## 4. Area-weighted substrate mixing

In real applications, an AOI rarely sits on a single uniform substrate. Pass an area-weighted dictionary and the package computes the area-weighted mean coefficient (manuscript § 3.4 convention):

In [ ]:
mixes = {
    "Pure reg":                           {"reg": 1.0},
    "70 % reg + 30 % wadi-bottom":        {"reg": 0.7, "wadi_bottom": 0.3},
    "50 % reg + 50 % wadi-bottom":        {"reg": 0.5, "wadi_bottom": 0.5},
    "Pure wadi-bottom":                   {"wadi_bottom": 1.0},
}

for label, mix in mixes.items():
    r = seri.compute(P=10.79, A=1624, season=2, substrate=mix)
    print(f"  {label:<35}  g = {r.g:.3f}   SERI = {r.value:>6.0f}   {r.tier_name}")

## 5. Sensitivity to the spatial exponent α (manuscript § 4.2)

The default `α = 0.68` is the working value pending formal calibration on the *n* ≈ 150 event archive (companion paper, in preparation). The manuscript reports that the *perennial-response* classification of Abadla 2015 is robust for α ∈ [0.66, 0.78].

In [ ]:
alphas = [0.50, 0.60, 0.66, 0.68, 0.72, 0.78, 0.85]

for alpha in alphas:
    r = seri.compute(P=10.79, A=1624, season=2, substrate="mixed", alpha=alpha)
    in_band = "✓ in published band" if 0.66 <= alpha <= 0.78 else ""
    print(f"  α = {alpha:.2f}   SERI = {r.value:>7.0f}   {r.tier.name:<10}  {in_band}")

## 6. The six ecological tiers

Use `seri.tier_table()` for a programmatically usable description of Table 1 of the manuscript:

In [ ]:
for name, lo, hi, label in seri.tier_table():
    bound = f"≥ {lo:.0f}" if hi == float('inf') else f"[{lo:.0f}, {hi:.0f})"
    print(f"  {name:<10}  {bound:<18}  {label}")

## 7. Optional: plotting

If `matplotlib` is installed (`pip install "seri[plot]"`), the `seri.plotting` module provides two ready-made figures.

In [ ]:
import matplotlib.pyplot as plt
from seri.plotting import plot_tier_bar, plot_event_summary

fig, ax = plt.subplots(figsize=(9, 1.8))
plot_tier_bar(result.value, label="Abadla 2015", ax=ax)
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))
plot_event_summary(result, ax=ax)
plt.tight_layout()
plt.show()

## Citation

If this notebook or the underlying package was useful for your research, please cite both the software and the concept paper:

**Software**
```
Selkh, C. (2026). SERI — Spatial Effective Rainfall Index
[Computer software]. Zenodo. https://doi.org/10.5281/zenodo.PLACEHOLDER_SERI
```

**Concept paper**
```
Selkh, C. (2026). A Century After De Martonne: Why Spatial Coherence
is the Missing Dimension of Aridity in the Hyper-Arid Sahara.
Earth-Science Reviews, in review.
```